# Recolección de datos: archivos, API y web scraping

**Fundamentos de Ciencia de Datos e IA (CDIAG1003)** · Semana 3 · Tema 2.3 · Demostración en la sesión 5 · Notebook de estudio en casa (sin nota)

Hasta ahora los datos llegaron listos: un archivo CSV limpio en la carpeta `datos`. En un proyecto real, **conseguir los datos** es la segunda etapa del ciclo de vida y suele ser la más incierta. Este notebook recorre las tres vías más comunes para traer datos de la web a un DataFrame.

| Sección | Vía | Herramientas | Lectura |
|---|---|---|---|
| 1 | Las tres vías y cómo elegir | — | — |
| 2 | **Archivo** descargado: CSV, Excel, JSON | `pd.read_csv`, `pd.read_excel`, `pd.read_json` | McKinney §6.1 y §6.2 |
| 3 | **API** web | `requests`, `pd.json_normalize` | McKinney §6.3 |
| 4 | ***Web scraping*** | `pd.read_html`, `requests`, Beautiful Soup | McKinney §6.1 «XML y HTML: raspado web» · documentación de Beautiful Soup |
| 5 | Bases de datos (solo mención) | `pd.read_sql` | McKinney §6.4 |

**En la sesión 5** el profesor ejecuta las celdas de demostración. **En casa**, después de la Lección 1 y antes de la sesión 8, ejecute el notebook completo y resuelva los cuatro ejercicios «Su turno»: 2.1, 3.1, 4.1 y 4.4.2. Con conexión, las celdas de la API (sección 3) y de la página real (sección 4.4) consultan internet; sin conexión, usan las copias guardadas. Este notebook es la base del apartado 1 de la Tarea 1 (el origen de los datos).

Todo el notebook funciona **sin conexión**: los archivos necesarios están en la carpeta `datos`, y las celdas que consultan internet usan una copia guardada cuando la red no responde.

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import requests                      # peticiones HTTP: pedir una página o consultar una API
from bs4 import BeautifulSoup        # Beautiful Soup: leer el HTML de una página

print("pandas", pd.__version__, "| requests", requests.__version__)

archivos = ["ventas_tienda.csv", "ventas_abril.csv", "tienda.xlsx", "estudiantes.json",
            "api_banco_mundial_poblacion.json", "api_banco_mundial_pib.json", "catalogo_tienda.html",
            "libros_pagina_1.html", "libros_pagina_2.html", "libros_pagina_3.html"]
for a in archivos:
    print("OK   " if os.path.exists("datos/" + a) else "FALTA", a)

> Si la celda anterior produce `ModuleNotFoundError`, falta una biblioteca. En Anaconda se instala desde *Anaconda Prompt* con `conda install requests beautifulsoup4 lxml openpyxl`. Si algún archivo aparece como `FALTA`, el notebook no está junto a la carpeta `datos` de la semana 3.

## 1. Las tres vías y cómo elegir

| Vía | Qué es | Cuándo usarla | Dificultad |
|---|---|---|---|
| **Archivo** | Alguien publica los datos para descargar (CSV, Excel, JSON) | Siempre que exista: portales de datos abiertos, INEC, Kaggle | baja |
| **API** | Un servicio que responde a una consulta con datos (casi siempre en JSON) | Datos que cambian, o cuando solo se necesita una parte | media |
| ***Web scraping*** | Un programa lee el HTML de una página y extrae los datos | Solo si no hay archivo ni API, y si el sitio lo permite | alta |

**Regla de orden:** primero se busca una descarga; después, una API; el *scraping* es el último recurso. Cuanto más abajo en la tabla, más código, más errores y más cuidado legal.

## 2. Vía 1 · Archivos

### 2.1 Un CSV que «no abre»

`ventas_tienda.csv` se lee con una línea. El archivo de las ventas de abril llegó exportado desde una hoja de cálculo configurada en español, y no se deja leer igual:

In [ ]:
try:
    pd.read_csv("datos/ventas_abril.csv")
except Exception as e:
    print(type(e).__name__, "→", e)

**Primer problema: la codificación (*encoding*).** Un archivo de texto guarda cada letra como uno o varios bytes, según una tabla de codificación. `read_csv` supone la tabla `utf-8`; este archivo usa `latin-1` (frecuente en archivos creados con Excel en Windows), y las letras con tilde no coinciden.

Antes de corregir, conviene **mirar el archivo crudo**:

In [ ]:
with open("datos/ventas_abril.csv", encoding="latin-1") as f:
    for _ in range(4):
        print(f.readline().rstrip())

Se ven tres diferencias más con el archivo de enero a marzo: el separador es **punto y coma**, los decimales usan **coma** (`18,0`) y las fechas están como **día/mes/año**. Se corrige un problema a la vez:

In [ ]:
paso1 = pd.read_csv("datos/ventas_abril.csv", encoding="latin-1")
print(paso1.shape)                     # (60, 1): una sola columna → el separador no es la coma

In [ ]:
paso2 = pd.read_csv("datos/ventas_abril.csv", encoding="latin-1", sep=";")
print(paso2.shape)
print(paso2.dtypes)                    # precio_unitario se leyó como texto: "18,0" no es un número

In [ ]:
paso3 = pd.read_csv("datos/ventas_abril.csv", encoding="latin-1", sep=";", decimal=",")
print(paso3.dtypes)                    # precio_unitario ya es float64; falta la fecha
paso3.head(3)

**La fecha.** Sin indicar el formato, `pd.to_datetime` deduce el formato a partir del **primer valor** y, ante la duda, supone **mes/día/año**: lee `03/04/2026` como 4 de marzo. Con una columna en formato día/mes/año hay dos resultados posibles:

- si **algún día es mayor que 12**, se produce un `ValueError` (el día 13 no puede ser un mes): el error se ve;
- si **todos los días son 12 o menos** (por ejemplo, un archivo con los primeros días del mes), **no hay mensaje** y el día y el mes quedan intercambiados: es un error silencioso.

La solución es la misma en los dos casos: indicar el formato.

In [ ]:
print(pd.to_datetime("03/04/2026"))                          # 4 de marzo (mes primero)
print(pd.to_datetime("03/04/2026", format="%d/%m/%Y"))       # 3 de abril (día primero)

# Las 8 primeras fechas del archivo (1 al 5 de abril): sin formato no hay mensaje, y los meses quedan mal
print(pd.to_datetime(paso3["fecha"].head(8)).dt.month.tolist())

# La columna completa tiene días mayores que 12: sin formato se produce un error
try:
    pd.to_datetime(paso3["fecha"])
except ValueError as error:
    print("ValueError:", str(error).split(".")[0])

| Síntoma | Parámetro de `read_csv` |
|---|---|
| `UnicodeDecodeError`, o letras extrañas como `SartÃ©n` | `encoding="latin-1"` (o `"utf-8"`) |
| Todo queda en una sola columna | `sep=";"` (o `"\t"` para tabulaciones) |
| Los decimales se leen como texto | `decimal=","` |
| Las fechas quedan como texto; al convertirlas, `ValueError` o día y mes intercambiados | `pd.to_datetime(columna, format="%d/%m/%Y")` |

**Su turno 2.1.** Lea `datos/ventas_abril.csv` en el DataFrame `abril`, con los tres parámetros necesarios. Convierta la columna `fecha` a fecha con el formato día/mes/año. Cree la columna `ingreso` y guarde en `ingreso_abril` el ingreso total de abril.

In [ ]:
# TODO: reemplace los ... por su código
abril = ...
abril["fecha"] = ...
abril["ingreso"] = ...
ingreso_abril = ...
print(abril.dtypes)
print(ingreso_abril)

In [ ]:
assert isinstance(abril, pd.DataFrame) and abril.shape[0] == 60 and "ciudad" in abril.columns, "Lea el archivo con encoding='latin-1' y sep=';'"
assert abril["precio_unitario"].dtype == float, "Falta decimal=',': el precio se leyó como texto"
assert pd.api.types.is_datetime64_any_dtype(abril["fecha"]), "Convierta fecha con pd.to_datetime(..., format='%d/%m/%Y')"
assert abril["fecha"].dt.month.eq(4).all(), "Todas las ventas son de abril: revise el formato de la fecha (día/mes/año)"
assert round(float(ingreso_abril), 1) == 2070.4, "ingreso = cantidad × precio_unitario; después, la suma"

### 2.2 Excel: varias hojas en un archivo

`pd.read_excel` lee una hoja de un libro de Excel. Con `sheet_name=None` devuelve **todas** las hojas, en un diccionario.

In [ ]:
productos = pd.read_excel("datos/tienda.xlsx", sheet_name="productos")
productos

In [ ]:
hojas = pd.read_excel("datos/tienda.xlsx", sheet_name=None)     # diccionario: nombre de la hoja → DataFrame
print(list(hojas.keys()))
hojas["locales"]

En la sesión 3 quedó una pregunta sin respuesta: «¿qué producto deja más ganancia?». Faltaba el **costo** de cada producto. Ese dato existía, pero en otro archivo: el de contabilidad.

In [ ]:
productos["margen"] = productos["precio_unitario"] - productos["costo_unitario"]
productos.sort_values("margen", ascending=False).head(3)

Para saber cuánta ganancia dejó cada **venta** hay que unir esta tabla con la de ventas. Unir tablas (`merge`) es tema de la sesión 9.

### 2.3 JSON

`pd.read_json` convierte un archivo JSON en DataFrame cuando el archivo es una lista de registros con las mismas claves. Es el `estudiantes.json` del Notebook 0.

In [ ]:
estudiantes = pd.read_json("datos/estudiantes.json")
estudiantes

La columna `notas` contiene **listas**: el archivo es semiestructurado (sesión 3) y no todo cabe en una celda simple. Convertir esas listas en columnas es parte de la preparación de datos.

### 2.4 Leer directamente desde una dirección web

`read_csv` acepta una URL en lugar de una ruta. La celda siguiente necesita conexión; si no la hay, muestra un aviso y el notebook continúa.

In [ ]:
url_csv = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
try:
    pinguinos = pd.read_csv(url_csv)
    print(pinguinos.shape)
    print(pinguinos.head(3))
except Exception:
    print("No se pudo leer el archivo desde internet: esta celda necesita conexión. El notebook continúa.")

## 3. Vía 2 · API web

Una **API** (*application programming interface*) web es una dirección a la que se le envía una consulta y que responde con datos, casi siempre en formato **JSON**. No hay página que mirar: la respuesta está hecha para programas.

Ejemplo: la API de datos abiertos del Banco Mundial. La consulta se arma con tres partes:

```
https://api.worldbank.org/v2/country/EC/indicator/SP.POP.TOTL?format=json&date=2015:2024
└────────── dirección base ─────────┘ └ país ┘ └── indicador ──┘ └────── parámetros ──────┘
```

`EC` es Ecuador y `SP.POP.TOTL` es el indicador «población total». Los parámetros van después del signo `?`, separados por `&`. Puede pegar esa dirección en el navegador para ver la respuesta.

Con `requests` se hace la misma consulta desde Python:

In [ ]:
url = "https://api.worldbank.org/v2/country/EC/indicator/SP.POP.TOTL"
parametros = {"format": "json", "date": "2015:2024"}

try:
    resp = requests.get(url, params=parametros, timeout=10)     # petición GET
    print("Código de estado:", resp.status_code)                # 200 = correcto; 404 = no existe; 429 = demasiadas peticiones
    resp.raise_for_status()                                     # detiene la ejecución si la respuesta es un error
    datos = resp.json()                                         # el JSON, convertido en listas y diccionarios de Python
    print("Datos recibidos de la API:", resp.url)
except requests.exceptions.RequestException:                    # error de red: sin conexión, sitio bloqueado, respuesta de error
    print("No se pudo consultar la API. Se usa la copia guardada: datos/api_banco_mundial_poblacion.json")
    with open("datos/api_banco_mundial_poblacion.json", encoding="utf-8") as f:
        datos = json.load(f)

**Explorar la respuesta antes de convertirla.** Cada API organiza su JSON a su manera; hay que leerlo (y leer su documentación).

In [ ]:
print(type(datos), len(datos))      # una lista con dos elementos
print(datos[0])                     # el primero: información sobre la respuesta (páginas, total de registros)

In [ ]:
print(len(datos[1]))                # el segundo: la lista de registros
datos[1][0]                         # un registro: un diccionario con diccionarios dentro

Cada registro tiene diccionarios **anidados** (`indicator`, `country`). `pd.json_normalize` los aplana: crea una columna por cada clave, con nombres como `country.value`.

In [ ]:
tabla = pd.json_normalize(datos[1])
print(tabla.columns.tolist())
tabla.head(3)

In [ ]:
poblacion = tabla[["country.value", "date", "value"]].rename(
    columns={"country.value": "pais", "date": "anio", "value": "poblacion"})
poblacion["anio"] = poblacion["anio"].astype(int)       # el año llegó como texto
poblacion = poblacion.sort_values("anio")
poblacion

In [ ]:
# comprobación de la tabla (no es un ejercicio)
assert poblacion.shape == (10, 3) and poblacion["anio"].tolist() == list(range(2015, 2025)), "Se esperan 10 años, de 2015 a 2024"
assert (poblacion["poblacion"] > 16_000_000).all(), "La población del Ecuador en esos años supera los 16 millones"
print("Tabla de población correcta")

La misma lógica, empaquetada en una función para reutilizarla:

In [ ]:
def pedir_json(url, parametros, copia):
    # Consulta una API y devuelve el JSON. Si la red no responde, usa el archivo de copia.
    try:
        resp = requests.get(url, params=parametros, timeout=10)
        resp.raise_for_status()
        print("Datos recibidos de la API:", resp.url)
        return resp.json()
    except requests.exceptions.RequestException:           # error de red: sin conexión, sitio bloqueado, respuesta de error
        print("No se pudo consultar la API. Se usa la copia guardada:", copia)
        with open(copia, encoding="utf-8") as f:
            return json.load(f)

**Buenas prácticas con una API:** leer su documentación y sus condiciones de uso; usar `timeout`; revisar el código de estado; no repetir la misma consulta sin necesidad (guardar la respuesta en un archivo); y, si la API exige una clave (*API key*), no escribirla en un notebook que se va a compartir.

**Su turno 3.1.** Consulte el indicador `NY.GDP.PCAP.CD` (PIB per cápita en dólares corrientes) de Ecuador, Colombia y Perú entre 2019 y 2023. En la dirección, varios países se separan con punto y coma: `country/EC;CO;PE`. Use la función `pedir_json` con la copia `datos/api_banco_mundial_pib.json`. Construya el DataFrame `pib` con las columnas `pais`, `anio` (entero) y `pib_per_capita`.

In [ ]:
# TODO: reemplace los ... por su código
url_pib = ...
parametros_pib = ...
datos_pib = pedir_json(url_pib, parametros_pib, "datos/api_banco_mundial_pib.json")

pib = ...
pib.head()

In [ ]:
assert isinstance(url_pib, str) and "NY.GDP.PCAP.CD" in url_pib and all(c in url_pib.upper() for c in ("EC", "CO", "PE")), "Revise la dirección: los tres países y el indicador"
assert isinstance(parametros_pib, dict) and parametros_pib.get("format") == "json" and str(parametros_pib.get("date")) == "2019:2023", "Revise los parámetros: format y date (2019:2023)"
assert isinstance(pib, pd.DataFrame) and list(pib.columns) == ["pais", "anio", "pib_per_capita"], "Columnas esperadas: pais, anio, pib_per_capita"
assert pib.shape[0] == 15 and set(pib["pais"]) == {"Ecuador", "Colombia", "Peru"}, "Tres países × cinco años = 15 filas"
assert pib["anio"].dtype.kind == "i" and set(pib["anio"]) == {2019, 2020, 2021, 2022, 2023}, "Convierta anio a entero con astype(int)"
print("Ejercicio 3.1 correcto")

## 4. Vía 3 · *Web scraping*

Cuando los datos solo están **en una página web**, un programa puede leer el HTML de la página y extraerlos. Antes de escribir código hay que entender cómo está hecha la página.

### 4.1 Anatomía de una página

Abra `datos/catalogo_tienda.html` con el navegador (doble clic en el archivo). Es la página del catálogo de la tienda del curso. Después, haga clic derecho sobre el nombre de un producto y elija **Inspeccionar**: el navegador muestra el HTML de ese elemento.

Una página HTML es texto con **etiquetas** (*tags*) anidadas. Cada elemento tiene esta forma:

```html
<span class="precio">$ 18,00</span>
```

- `span` es la **etiqueta**: indica el tipo de elemento (`h1`, `p`, `a`, `table`, `tr`, `td`, `div`, `span`, `article`…).
- `class="precio"` es un **atributo**. Los más útiles para encontrar elementos son `class` (se repite en elementos parecidos) e `id` (único en la página).
- `$ 18,00` es el **texto** del elemento.

Esta es una tarjeta de producto de la página:

```html
<article class="producto" data-categoria="Hogar" data-codigo="H-01">
  <h3>Sartén</h3>
  <span class="precio">$ 18,00</span>
  <span class="estado disponible">Disponible</span>
  <p class="valoracion" data-estrellas="4">★★★★☆</p>
  <a class="detalle" href="productos/sarten.html">Ver detalle</a>
</article>
```

Hay diez tarjetas iguales en su estructura. Esa repetición es lo que permite extraer los datos con un programa.

In [ ]:
with open("datos/catalogo_tienda.html", encoding="utf-8") as f:
    html = f.read()

print(len(html), "caracteres")
print(html[html.find("<article"):html.find("</article>") + 10])      # la primera tarjeta, tal como está en el archivo

### 4.2 El atajo: `pd.read_html` para tablas

Si los datos están en una **tabla** HTML (etiqueta `<table>`), `pd.read_html` hace todo el trabajo: devuelve una **lista** con un DataFrame por cada tabla de la página.

In [ ]:
tablas = pd.read_html("datos/catalogo_tienda.html", encoding="utf-8")
print(len(tablas), "tabla(s) en la página")
locales = tablas[0]
locales

`read_html` también acepta una URL. Solo encuentra tablas: los productos de la página no están en una tabla, sino en tarjetas, y para leerlos hace falta Beautiful Soup.

### 4.3 Beautiful Soup: buscar elementos por etiqueta, clase o id

`BeautifulSoup` convierte el texto HTML en un objeto en el que se puede buscar.

| Para buscar… | Se escribe |
|---|---|
| el primer elemento con una etiqueta | `soup.find("h1")` |
| todos los elementos con una etiqueta y una clase | `soup.find_all("article", class_="producto")` |
| un elemento por su `id` | `soup.find(id="titulo")` |
| el texto de un elemento | `elemento.text` |
| el valor de un atributo | `elemento["href"]` |

In [ ]:
soup = BeautifulSoup(html, "html.parser")

print(soup.title.text)                                   # el contenido de <title>
print(soup.find("h1").text)                              # el primer <h1>
print(soup.find(id="fecha-actualizacion").text)          # el elemento con ese id

In [ ]:
tarjetas = soup.find_all("article", class_="producto")   # una lista con todas las tarjetas
print(len(tarjetas), "productos")

t = tarjetas[0]                                          # se estudia una tarjeta antes de recorrerlas todas
print(t.find("h3").text)
print(t.find("span", class_="precio").text)
print(t["data-categoria"])                               # atributo de la propia tarjeta
print(t.find("a")["href"])                               # atributo de un elemento interior

Lo que funciona con una tarjeta se repite con todas. Aquí el bucle es apropiado: se recorre una lista de elementos de la página, no una columna de números.

In [ ]:
filas = []
for t in tarjetas:
    filas.append({
        "producto": t.find("h3").text,
        "categoria": t["data-categoria"],
        "precio_txt": t.find("span", class_="precio").text,
        "estado": t.find("span", class_="estado").text,
        "estrellas": int(t.find("p", class_="valoracion")["data-estrellas"]),
    })

catalogo = pd.DataFrame(filas)
catalogo

El precio llegó como **texto** (`"$ 18,00"`): lo que se extrae de una página casi siempre necesita limpieza. Los métodos `.str` de pandas trabajan sobre el texto de toda la columna (se estudian en la sesión 7).

In [ ]:
catalogo["precio"] = (catalogo["precio_txt"]
                      .str.replace("$", "", regex=False)      # quitar el símbolo
                      .str.replace(",", ".", regex=False)     # coma decimal → punto
                      .str.strip()                            # quitar espacios
                      .astype(float))
print(catalogo.dtypes)
print(catalogo[["producto", "precio"]].head(3))

In [ ]:
# comprobación: los precios extraídos de la página coinciden con los del archivo de ventas
ventas = pd.read_csv("datos/ventas_tienda.csv")
precios_ventas = ventas.groupby("producto")["precio_unitario"].first()
precios_pagina = catalogo.set_index("producto")["precio"]
assert (precios_pagina.sort_index() == precios_ventas.sort_index()).all(), "Los precios extraídos deben coincidir con los del archivo de ventas"
print("Los 10 precios extraídos de la página coinciden con los de ventas_tienda.csv")

**Selectores CSS.** `soup.select` busca con la misma notación que usan las hojas de estilo: `etiqueta`, `.clase`, `#id`, y un espacio para «dentro de».

In [ ]:
print([e.text for e in soup.select("article.producto h3")][:4])      # los <h3> dentro de <article class="producto">
print(len(soup.select("#tabla-locales tbody tr")))                    # filas del cuerpo de la tabla con id tabla-locales
print([e.text for e in soup.select("span.agotado")])                  # los <span> con la clase agotado

**Su turno 4.1.** Con `tarjetas` (o con `soup.select`), obtenga:

- `codigos`: una lista con el atributo `data-codigo` de las diez tarjetas;
- `agotados`: una lista con los **nombres** de los productos cuyo estado es «Agotado»;
- `primer_enlace`: el atributo `href` del enlace de la primera tarjeta.

In [ ]:
# TODO: reemplace los ... por su código
codigos = ...
agotados = ...
primer_enlace = ...
print(codigos)
print(agotados)
print(primer_enlace)

In [ ]:
assert isinstance(codigos, list) and len(codigos) == 10 and codigos[0] == "H-01" and codigos[-1] == "P-02", "Recorra las tarjetas y lea el atributo: t['data-codigo']"
assert agotados == ["Audífonos", "Bolígrafo"], "Nombre (h3) de las tarjetas cuyo span de estado tiene el texto Agotado"
assert primer_enlace == "productos/sarten.html", "tarjetas[0].find('a')['href']"
print("Ejercicio 4.1 correcto")

### 4.4 Una página real, con paginación

Con una página publicada en internet, el único paso nuevo es **pedir** el HTML con `requests`; lo demás es igual. El sitio `books.toscrape.com` es una librería ficticia creada para practicar *web scraping*: tiene 1 000 libros repartidos en 50 páginas de 20 libros, con direcciones que solo cambian en un número (`page-1.html`, `page-2.html`…).

La celda siguiente lee las **tres primeras páginas**. Observe tres detalles: la cabecera `User-Agent` identifica quién hace la petición; `resp.encoding = "utf-8"` evita que el símbolo `£` se lea mal; y `time.sleep(1)` espera un segundo entre peticiones para no cargar el servidor.

Como en la sección 3, si la red no responde (o si la red del lugar bloquea el sitio), la función `pedir_pagina` usa una **copia guardada** de cada página: `datos/libros_pagina_1.html`, `_2` y `_3`. Las copias se guardaron el 11 de octubre de 2026 y conservan la lista de libros con las mismas etiquetas y clases de la página original. La celda indica de dónde salió cada página.

In [ ]:
URL_PAGINA = "https://books.toscrape.com/catalogue/page-{}.html"
cabeceras = {"User-Agent": "Curso CDIAG1003 ESPOL - practica de clase"}


def pedir_pagina(pagina):
    # Pide una página del sitio y devuelve su HTML. Si la red no responde, usa la copia guardada.
    try:
        resp = requests.get(URL_PAGINA.format(pagina), headers=cabeceras, timeout=10)
        resp.raise_for_status()                                  # se detiene si la respuesta es un error
        resp.encoding = "utf-8"
        time.sleep(1)                                            # pausa entre peticiones
        print("Página", pagina, "recibida de internet:", resp.url)
        return resp.text
    except requests.exceptions.RequestException:              # error de red: sin conexión, sitio bloqueado, respuesta de error
        copia = "datos/libros_pagina_" + str(pagina) + ".html"
        print("Página", pagina, "no se pudo pedir al sitio. Se usa la copia guardada:", copia)
        with open(copia, encoding="utf-8") as f:
            return f.read()


libros = []
for pagina in range(1, 4):                                       # páginas 1, 2 y 3
    sopa = BeautifulSoup(pedir_pagina(pagina), "html.parser")
    for art in sopa.find_all("article", class_="product_pod"):
        libros.append({
            "titulo": art.find("h3").find("a")["title"],
            "precio_txt": art.find("p", class_="price_color").text,
            "estrellas": art.find("p", class_="star-rating")["class"][1],
            "pagina": pagina,
        })
libros = pd.DataFrame(libros)
print(libros.shape)
libros.head()

**Su turno 4.4.2.** Con el DataFrame `libros` de la celda anterior: (a) cree la columna `precio` como número (quite el símbolo `£` y convierta a `float`); (b) calcule el precio promedio de los 60 libros; (c) cuente cuántos libros hay de cada número de estrellas (`value_counts`). Escriba el código en la celda siguiente; este ejercicio no tiene `assert`: compare su resultado con el de la versión con soluciones.

In [ ]:
# TODO: escriba aquí su código

### 4.5 Ética y legalidad: antes de extraer datos de un sitio

Que una página sea pública no significa que todo uso esté permitido. Lista de verificación antes de escribir un *scraper*:

1. **¿Hay una descarga o una API?** Si existe, se usa esa vía.
2. **Términos de uso del sitio.** Muchos sitios prohíben la extracción automática; hay que leerlos.
3. **`robots.txt`.** Archivo en la raíz del sitio (por ejemplo, `https://es.wikipedia.org/robots.txt`) que indica qué partes no deben recorrer los programas automáticos. Se respeta.
4. **Ritmo.** Pocas peticiones, con pausas (`time.sleep`), y fuera de las horas de mayor uso. Un programa puede pedir en un minuto lo que una persona en un día: saturar un servidor causa daño.
5. **Datos personales.** No se recolectan nombres, correos, teléfonos ni perfiles de personas sin una base legal. En el Ecuador, el tratamiento de datos personales está regulado por la Ley Orgánica de Protección de Datos Personales (2021).
6. **Derechos de autor.** Textos, imágenes y bases de datos pueden estar protegidos: extraerlos para analizarlos no autoriza a republicarlos.
7. **Identificarse y guardar.** Usar un `User-Agent` que diga quién es, y guardar lo descargado en un archivo para no volver a pedirlo.

Para el proyecto integrador: si el grupo obtiene sus datos por *scraping*, el notebook debe indicar el sitio, la fecha de extracción y por qué el uso está permitido.

## 5. Bases de datos (solo mención)

Muchas organizaciones guardan sus datos en bases de datos relacionales. pandas las consulta con `pd.read_sql("SELECT …", conexion)`, que devuelve un DataFrame. Se necesita una conexión (por ejemplo, con `sqlite3` o SQLAlchemy) y conocer el lenguaje SQL, que se estudia en otros cursos de la carrera. Referencia: McKinney §6.4.

## Resumen

| Los datos están en… | Vía | Herramienta | Atención a… |
|---|---|---|---|
| un archivo CSV | archivo | `pd.read_csv` | `sep`, `encoding`, `decimal`, formato de fecha |
| un libro de Excel | archivo | `pd.read_excel` | `sheet_name`; hojas con títulos o celdas combinadas |
| un archivo JSON | archivo | `pd.read_json`, `pd.json_normalize` | diccionarios y listas anidados |
| una API | API | `requests.get(...).json()` + `pd.json_normalize` | documentación, código de estado, límites de uso |
| una tabla en una página | *scraping* | `pd.read_html` | devuelve una lista de tablas |
| otros elementos de una página | *scraping* | `requests` + `BeautifulSoup` | permisos, ritmo, limpieza del texto |
| una base de datos | consulta | `pd.read_sql` | conexión y SQL |

### Errores frecuentes

| Síntoma | Causa | Solución |
|---|---|---|
| `UnicodeDecodeError`, o texto como `SartÃ©n` | codificación equivocada | `encoding="latin-1"` o `encoding="utf-8"` |
| Una sola columna con todo el texto | separador equivocado | `sep=";"` |
| `ValueError: time data … doesn't match format`, o día y mes intercambiados | fecha en formato día/mes/año convertida sin indicar el formato | `format="%d/%m/%Y"` |
| `ConnectionError`, `ProxyError` o `Timeout` | sin red, la red del lugar bloquea el sitio, o el sitio no responde | revisar la conexión; usar la copia guardada |
| Código de estado 403, 404 o 429 | acceso no permitido, dirección equivocada o demasiadas peticiones | revisar la URL y las condiciones de uso; esperar |
| `AttributeError: 'NoneType' object has no attribute 'text'` | `find` no encontró el elemento y devolvió `None` | revisar la etiqueta y la clase con **Inspeccionar** |
| `find_all` devuelve una lista vacía | la clase no coincide, o la página carga sus datos con JavaScript | revisar el HTML recibido: `print(resp.text[:500])` |